## 04 - silver -> gold

Rebuilds the reporting tables from the current silver rows. Full overwrite - the data is small, and every run is a new Delta version so older states are still reachable with time travel.

| table | grain |
|---|---|
| gold_dim_patient | one row per patient, plus placeholder rows for patients that facts point to but that weren't loaded yet |
| gold_dim_date | calendar covering all fact dates |
| gold_fact_encounter | one row per encounter |
| gold_fact_observation | one row per measured value (blood pressure panel -> one row per component) |
| gold_fact_condition | one row per condition |

Views for the SQL analytics endpoint are in `sql/gold_views.sql`.

In [ ]:
run_id = ""
config_path = "/lakehouse/default/Files/config/pipeline_config.json"
code_path = "/lakehouse/default/Files/code"
lakehouse_root = ""    # blank: use the default lakehouse attached to the notebook

In [ ]:
import json
import sys

if code_path and code_path not in sys.path:
    sys.path.insert(0, code_path)

from pyspark.sql import functions as F

from fhir_lakehouse.config import apply_lakehouse_root, ctl_table, gold_table, load_config
from fhir_lakehouse.control import StepLog, new_run_id
from fhir_lakehouse.gold import build_gold
from fhir_lakehouse.storage import Lakehouse, get_spark

cfg = apply_lakehouse_root(load_config(config_path), lakehouse_root)
run_id = run_id or new_run_id()

spark = get_spark()
spark.conf.set("spark.sql.session.timeZone", cfg["spark"]["timezone"])
lh = Lakehouse(spark, cfg)

In [ ]:
step = StepLog(lh, cfg, run_id, "silver_to_gold")
try:
    stats = build_gold(lh, cfg, run_id)
    step.finish("SUCCEEDED", rows_written=sum(stats.values()), details=stats)
except Exception as e:
    step.finish("FAILED", error=repr(e))
    raise

In [ ]:
# every fact row should find its patient
dim = lh.read(gold_table(cfg, "dim_patient")).select("patient_id")
for fact in ("fact_encounter", "fact_observation", "fact_condition"):
    orphans = (lh.read(gold_table(cfg, fact)).where(F.col("patient_id").isNotNull())
                 .join(dim, "patient_id", "left_anti").count())
    print(f"{fact}: {orphans} rows without a patient in dim_patient")

(lh.read(ctl_table(cfg, "load_log")).where(F.col("run_id") == run_id)
   .select("step", "resource_type", "status", "rows_read", "rows_written", "rows_inserted", "rows_updated", "duration_sec")
   .orderBy("started_at")
   .show(50, truncate=False))

In [ ]:
result = json.dumps(stats, default=str)
print(result)
try:
    notebookutils.notebook.exit(result)  # shows up as the activity output in the pipeline
except NameError:
    pass  # not running in Fabric